## Import libraries and dataset
## Separate x_train and y_train

In [85]:
import pandas as pd
from pathlib import Path
import torch
import numpy as np
pd.set_option('display.max_columns', None)

notebook_dir = Path('.').resolve()
project_root = notebook_dir.parent.parent.parent
file_path = project_root / "data" / "perceptron_toydata-truncated.txt"
df = pd.read_csv(file_path,sep="\t")


In [86]:
df.head(5)

,x1,x2,label
0,0.77,-1.14,0
1,-0.33,1.44,0
2,0.91,-3.07,0
3,-0.37,-1.91,0
4,-0.63,-1.53,0


In [87]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      20 non-null     float64
 1   x2      20 non-null     float64
 2   label   20 non-null     int64  
dtypes: float64(2), int64(1)
memory usage: 612.0 bytes


In [88]:
X_train = df[["x1", "x2"]].values
y_train = df["label"].values
X_train = torch.tensor(X_train).float()
y_train= torch.tensor(y_train)

In [89]:
X_train.shape

torch.Size([20, 2])

In [90]:
y_train.shape

torch.Size([20])

In [91]:
np.bincount(y_train)

array([10, 10])

## Implement Perceptron model

In [92]:
class Perceptron:
    def __init__(self, num_features):
        self.weights = torch.zeros(num_features).to(torch.float)
        self.bias = torch.tensor(0.)

    def forward(self, x):
        weighted_sum_z = torch.dot(x, self.weights) + self.bias
        return torch.where(weighted_sum_z>0.,torch.tensor(1.),torch.tensor(0.))
        
    
    def update(self, x, true_y):
        prediction = self.forward(x)
        error = true_y - prediction

        # update
        self.bias += error
        self.weights += error * x

        return error

In [93]:
pcp = Perceptron(num_features=2)

In [94]:
pcp.weights

tensor([0., 0.])

In [95]:
pcp.bias

tensor(0.)

In [96]:
pcp = Perceptron(num_features=2)
x = [1.1,2.1]
t_x = torch.tensor(x)
pcp.forward(t_x)

tensor(0.)

In [97]:
pcp.update(t_x,true_y=1)

tensor(1.)

In [98]:
print("model params")
print("weights : ",pcp.weights)
print("bias:", pcp.bias)

model params
weights :  tensor([1.1000, 2.1000])
bias: tensor(1.)


## Model training and evaluation

In [101]:
def train(model,all_x,all_y,epochs):
    for epoch in range(epochs):
        error_count = 0
        for x,y in zip(all_x,all_y):
            error = model.update(x,y)
            error_count += abs(error)
        print(f"epoch {epoch+1} errors {error_count}")

In [102]:
pcp = Perceptron(num_features=2)
train(model=pcp,all_x=X_train,all_y=y_train,epochs=5)

epoch 1 errors 1.0
epoch 2 errors 3.0
epoch 3 errors 1.0
epoch 4 errors 0.0
epoch 5 errors 0.0


In [103]:
def evaluation(model,all_x,all_y):
    correct =0.0
    for x, y in zip(all_x, all_y):
        prediction = model.forward(x)
        correct += int(prediction == y)

    return correct / len(all_y)



In [104]:
train_acc = evaluation(pcp, X_train, y_train)
train_acc

1.0